TASK 01 (BFS)

In [ ]:
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.h = 0

    def __lt__(self, other):
        return self.h < other.h

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def best_first_search(maze, start, goal):
    rows, cols = len(maze), len(maze[0])
    frontier = PriorityQueue()
    start_node = Node(start)
    start_node.h = heuristic(start, goal)
    frontier.put(start_node)

    visited = set()

    while not frontier.empty():
        current = frontier.get()
        if current.position == goal:
            path = []
            while current:
                path.append(current.position)
                current = current.parent
            return path[::-1]

        visited.add(current.position)

        for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
            new_pos = (current.position[0] + dx, current.position[1] + dy)

            if (0 <= new_pos[0] < rows and
                0 <= new_pos[1] < cols and
                maze[new_pos[0]][new_pos[1]] == 0 and
                new_pos not in visited):

                node = Node(new_pos, current)
                node.h = heuristic(new_pos, goal)
                frontier.put(node)

    return None


def multi_goal_navigation(maze, start, goals):
    current = start
    full_path = []

    remaining_goals = goals.copy()

    while remaining_goals:
        nearest = min(remaining_goals, key=lambda g: heuristic(current, g))
        path = best_first_search(maze, current, nearest)

        if not path:
            return None

        full_path.extend(path[:-1])
        current = nearest
        remaining_goals.remove(nearest)

    full_path.append(current)
    return full_path


maze = [
    [0,0,0,0,0],
    [0,1,1,0,0],
    [0,0,0,0,0],
    [0,1,0,1,0],
    [0,0,0,0,0]
]

start = (0,0)
goals = [(4,4), (2,2), (0,4)]

path = multi_goal_navigation(maze, start, goals)
print("Path covering all goals:", path)

Path covering all goals: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4), (3, 4), (2, 4), (1, 4), (0, 4)]


TASK 02 (A*)

In [ ]:
import random

def dynamic_a_star(graph, start, goal, heuristic):
    frontier = [(start, heuristic[start])]
    g_cost = {start: 0}
    came_from = {start: None}
    visited = set()

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current, _ = frontier.pop(0)

        if current == goal:
            path = []
            while current:
                path.append(current)
                current = came_from[current]
            return path[::-1]

        visited.add(current)

        for node in graph:
            for neighbor in graph[node]:
                if random.random() < 0.2:
                    graph[node][neighbor] = random.randint(1,10)

        for neighbor, cost in graph[current].items():
            new_g = g_cost[current] + cost
            f = new_g + heuristic[neighbor]

            if neighbor not in g_cost or new_g < g_cost[neighbor]:
                g_cost[neighbor] = new_g
                came_from[neighbor] = current
                frontier.append((neighbor, f))

    return None


graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'D': 5},
    'C': {'D': 2},
    'D': {'E': 4},
    'E': {}
}

heuristic = {'A':7,'B':6,'C':2,'D':1,'E':0}

path = dynamic_a_star(graph, 'A', 'E', heuristic)
print("Dynamic A* Path:", path)

Dynamic A* Path: ['A', 'C', 'D', 'E']


TASK 03 (GREEDY BFS)

In [ ]:
def greedy_delivery(graph, start, deliveries, heuristic):
    current = start
    time = 0
    route = [start]

    remaining = deliveries.copy()

    while remaining:
        # Sort by deadline first, then distance
        remaining.sort(key=lambda x: (x['deadline'], heuristic[current][x['node']]))

        next_delivery = remaining.pop(0)
        node = next_delivery['node']
        travel_cost = heuristic[current][node]

        time += travel_cost

        if time > next_delivery['deadline']:
            print("Missed delivery at", node)

        route.append(node)
        current = node

    return route


graph = {
    'Depot': {'A':2, 'B':4, 'C':3},
    'A': {}, 'B': {}, 'C': {}
}

heuristic = {
    'Depot': {'A':2,'B':4,'C':3},
    'A': {'B':2,'C':4},
    'B': {'A':2,'C':1},
    'C': {'A':4,'B':1}
}

deliveries = [
    {'node':'A', 'deadline':5},
    {'node':'B', 'deadline':3},
    {'node':'C', 'deadline':7}
]

route = greedy_delivery(graph, 'Depot', deliveries, heuristic)
print("Optimized Delivery Route:", route)

Missed delivery at B
Missed delivery at A
Missed delivery at C
Optimized Delivery Route: ['Depot', 'B', 'A', 'C']
